# Previsão de atrasos Olist: modelos, pipelines e avaliação temporal

**Objetivo:** comparar alternativas no mesmo conjunto tratado, selecionar pela validação e verificar **F1 macro ≥ 0,5 no teste real**. A meta foi confirmada pelo usuário como macro; ela não significa recall ou F1 da classe atrasado de 50%.

Este estudo é independente do pipeline da aplicação. Não altera `dataset_ml.csv`, `datasets_engineering.csv`, PostgreSQL, API, assistente ou frontend. O avaliador principal foi ajustado às métricas solicitadas; a implantação dos modelos estudados permanece separada.

**Como usar:** selecione `.venv-pipeline` e execute as células em ordem. A primeira execução treina os candidatos; as seguintes recuperam checkpoints compatíveis. Para outro estudo, altere `EXECUCAO` abaixo. As funções completas estão neste notebook, inclusive geração sintética, seleção, exportação e gráficos. As células de implementação podem ser expandidas.

**Saídas:** um CSV em `Dados/resultados_modelos/<EXECUCAO>.csv`; pacotes escolhidos em `Modelos/<EXECUCAO>/`; nove conjuntos real/sintético/misto em `Dados/<EXECUCAO com modelos substituído por conjuntos>/`; gráficos em `src/eda/graficos/<EXECUCAO>/`. Checkpoints e progresso ficam em `.cache/<EXECUCAO>/`.

**Retomada:** cada candidato concluído salva modelo, scores de treino/validação e metadados. Basta executar novamente: os ajustes concluídos não se repetem. A seleção congelada impede reutilizar o mesmo teste para escolher novos candidatos. Alterações de dados/cortes/hiperparâmetros exigem outra execução após o congelamento. Não é necessária uma automação recorrente.

In [ ]:
# Configuração do estudo. Para uma nova comparação, use outro nome de execução.
EXECUCAO = 'eda_modelos_20261116'
SEED = 42
THREADS = 4
LIMITE_AJUSTE_CTGAN = 20000
N_SINTETICOS = {'treino': 10000, 'validacao': 3000, 'teste': 3000}
GAN_CONFIG = {'epochs': 30, 'batch_size': 500, 'pac': 10,
              'generator_dim': (128, 128), 'discriminator_dim': (128, 128),
              'verbose': True, 'enable_gpu': False}
# Os cortes vêm de src/configuracoes/2_treinamento.json.
# Catálogo e grades de candidatos estão nas células de implementação abaixo.
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, message=".*generic.*timedelta.*")
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 115, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. O que pode ser previsto em cada momento

| Contexto | Quando a decisão acontece | Pedidos avaliados | Informação adicional |
|---|---|---|---|
| `atual_retrospectivo` | referência à aprovação | CSV tratado, incluindo PLN | status final, nota global e transportadora aleatória: diagnóstico retrospectivo |
| `aprovacao` | aprovação | pedidos com entrega observada e datas consistentes | características do pedido e prazos conhecidos |
| `despacho` | entrega ao transportador, antes do prazo | pedidos despachados antes do prazo | tempo de preparação e margem restante |
| `monitor_7` | sete dias antes do prazo | aprovados e ainda não entregues | situação conhecida e tempo em trânsito |
| `monitor_3` | três dias antes do prazo | aprovados e ainda não entregues | idem |
| `monitor_1` | um dia antes do prazo | aprovados e ainda não entregues | idem |

Os contextos têm populações diferentes. Um resultado alto no monitoramento não é uma previsão igualmente boa na aprovação. A regra de referência (`dummy`) mostra quanto se ganha apenas por selecionar pedidos ainda pendentes perto do prazo.

**Alvo mantido:** `delay_in_days = entrega − prazo` em dias fracionários; `delay_status = 1` quando essa diferença é **≥ 1 dia**. Valores negativos significam antecedência. Não se substituem dias desconhecidos por zero. O cenário operacional usa rótulos observados por datas; PLN permanece no diagnóstico retrospectivo, pois a duração e a confirmação do desfecho não estão disponíveis para esses casos.

**Disponibilidade dos atributos:** status final, nota do próprio pedido, nota global do vendedor e transportadora sorteada são excluídos das previsões operacionais. O histórico de atrasos e as notas anteriores são calculados somente com eventos registrados antes da decisão; a nota do próprio pedido é excluída. O histórico é atualizado com entregas/avaliações que passam a ser conhecidas ao longo do tempo, inclusive durante validação/teste, sem reajustar o estimador. Isso exige histórico operacional atualizado quando houver implantação.

**Divisão:** tempo da decisão `< 01/03/2018` é treino; de `01/03` até antes de `01/05` é validação; a partir de `01/05` é teste. No experimento operacional, rótulos do treino precisam estar conhecidos antes de março, e os da validação antes de maio. Essas restrições só se aplicam à cópia deste estudo. O CSV original e sua divisão não são alterados. A restrição de disponibilidade pode excluir atrasos longos próximos ao corte, introduzindo censura que deve ser considerada na interpretação.

**Limites:** o recorte operacional é retrospectivo de pedidos com entrega observada; não representa cancelamentos ou pedidos nunca entregues. A análise geográfica anterior descreveu toda a base, portanto este teste não é uma amostra externa inteiramente cega. Ainda assim, nenhuma métrica de desempenho de teste é usada para ajustar modelos, limiares ou escolher o vencedor.

In [ ]:
"""Código do estudo, a incorporar integralmente no notebook de comparação."""
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, warnings, os
import numpy as np
import pandas as pd
import joblib
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (f1_score,precision_score,recall_score,accuracy_score,confusion_matrix,
    mean_absolute_error,mean_squared_error,r2_score)
from threadpoolctl import threadpool_limits
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

RAIZ=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Dados/dataset_ml.csv').exists())
CFG=json.loads((RAIZ/'src/configuracoes/2_treinamento.json').read_text(encoding='utf-8'))
EXECUCAO=globals().get('EXECUCAO','eda_modelos_20261116')
SEED=globals().get('SEED',42)
THREADS=globals().get('THREADS',4)
CRITERIO='f1_macro'
VERSION='estudo_v1'
CACHE=RAIZ/'.cache'/EXECUCAO
CACHE.mkdir(parents=True,exist_ok=True)
FIGURAS=RAIZ/'src/eda/graficos'/EXECUCAO
FIGURAS.mkdir(parents=True,exist_ok=True)
VAL=pd.Timestamp(CFG['cortes']['validacao'])
TEST=pd.Timestamp(CFG['cortes']['teste'])
FONTES=[RAIZ/'Dados/dataset_ml.csv',RAIZ/'Dados/datasets_engineering.csv',
        RAIZ/'Dados/Brazilian Logistic/olist_orders_dataset.csv',RAIZ/'Dados/Brazilian Logistic/olist_order_items_dataset.csv']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
HASHES={str(p.relative_to(RAIZ)):sha(p) for p in FONTES}
FINGERPRINT=hashlib.sha256(json.dumps({'hashes':HASHES,'cortes':CFG['cortes'],
    'version':VERSION if SEED==42 else VERSION+'_seed'+str(SEED)},sort_keys=True).encode()).hexdigest()
def gravar_json(p,valor):
    p=Path(p);tmp=p.with_suffix(p.suffix+'.tmp')
    tmp.write_text(json.dumps(valor,indent=2,ensure_ascii=False,default=str),encoding='utf-8');tmp.replace(p)
def progresso(etapa,**extra):
    gravar_json(CACHE/'progresso.json',{'etapa':etapa,'atualizado_utc':datetime.now(timezone.utc).isoformat(),'fingerprint':FINGERPRINT,**extra})
    print(etapa,extra,flush=True)

REGIOES={'SP':'SP',**dict.fromkeys(['MG','RJ','ES'],'Sudeste_sem_SP'),**dict.fromkeys(['RS','SC','PR'],'Sul'),
    **dict.fromkeys(['BA','SE','AL','PE','PB','RN','CE','PI','MA'],'Nordeste'),
    **dict.fromkeys(['AM','AC','RR','RO','PA','AP','TO'],'Norte'),**dict.fromkeys(['DF','GO','MT','MS'],'Centro_Oeste')}
CAT_BASE=['customer_state','seller_state','predominant_product_category_name']
NUM_BASE=['price','freight_value','qtd_items','total_volume_cm3','total_weight','distance_customer_seller','approval_date_in_days']
ATUAL=CFG['atributos']

def preparar_base():
    d=pd.read_csv(FONTES[0],low_memory=False)
    assert d.order_id.is_unique and d.delay_status.isin([0,1]).all()
    e=pd.read_csv(FONTES[1],usecols=['order_id','seller_id','customer_lat','customer_lng','seller_lat','seller_lng'])
    o=pd.read_csv(FONTES[2],usecols=['order_id','order_purchase_timestamp','order_delivered_carrier_date',
        'order_delivered_customer_date','order_estimated_delivery_date'])
    it=pd.read_csv(FONTES[3],usecols=['order_id','shipping_limit_date'])
    it['shipping_limit_date']=pd.to_datetime(it.shipping_limit_date)
    it=it.groupby('order_id',as_index=False).shipping_limit_date.max()
    d=d.merge(e,on='order_id',validate='one_to_one').merge(o,on='order_id',validate='one_to_one').merge(it,on='order_id',how='left',validate='one_to_one')
    assert len(d)==len(pd.read_csv(FONTES[0],usecols=['order_id']))
    for c in ['order_approved_at','order_purchase_timestamp','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date','shipping_limit_date']:
        d[c]=pd.to_datetime(d[c],errors='coerce')
    d['y']=d.delay_status.astype(int)
    d['grupo_regiao']=d.customer_state.map(REGIOES).fillna('Nao_informada')
    d['faixa_distancia']=np.where(d.distance_customer_seller.gt(500),'acima_500','ate_500')
    d['grupo_regiao_distancia']=d.grupo_regiao+'__'+d.faixa_distancia
    d['rota_uf']=d.seller_state+'_'+d.customer_state
    d['prazo_prometido_dias']=(d.order_estimated_delivery_date-d.order_approved_at).dt.total_seconds()/86400
    d['limite_despacho_dias']=(d.shipping_limit_date-d.order_approved_at).dt.total_seconds()/86400
    d['margem_limite_ate_prazo']=(d.order_estimated_delivery_date-d.shipping_limit_date).dt.total_seconds()/86400
    d['compra_mes']=d.order_purchase_timestamp.dt.month
    d['compra_dia_semana']=d.order_purchase_timestamp.dt.dayofweek
    d['compra_hora']=d.order_purchase_timestamp.dt.hour
    d['compra_dia_mes']=d.order_purchase_timestamp.dt.day
    d['mesma_uf']=d.customer_state.eq(d.seller_state).astype(int)
    d['frete_preco']=d.freight_value/(d.price+1)
    d['volume_por_item']=d.total_volume_cm3/d.qtd_items.clip(lower=1)
    d['peso_por_item']=d.total_weight/d.qtd_items.clip(lower=1)
    for c in ['price','freight_value','total_weight','total_volume_cm3','distance_customer_seller']:
        d['log_'+c]=np.log1p(d[c].clip(lower=0))
    for c in ['customer_lat','seller_lat']:d.loc[~d[c].between(-34,6),c]=np.nan
    for c in ['customer_lng','seller_lng']:d.loc[~d[c].between(-74,-34),c]=np.nan
    observados=d.delay_status_source.eq('datas') & d.order_delivered_customer_date.notna()
    consistentes=(d.order_delivered_customer_date.ge(d.order_approved_at) & d.order_estimated_delivery_date.gt(d.order_approved_at))
    d['operacional']=observados & consistentes
    d=d.sort_values(['order_approved_at','order_id']).reset_index(drop=True)
    auditoria={'pedidos_csv':len(d),'rotulos_pln':int(d.delay_status_source.eq('pln').sum()),
        'observados_operacionais':int(d.operacional.sum()),'datas_inconsistentes_observadas':int((observados & ~consistentes).sum())}
    gravar_json(CACHE/'auditoria_fontes.json',{'hashes':HASHES,'contagens':auditoria,'cortes':CFG['cortes']})
    return d

def adicionar_historico(tabela,base):
    """Resultados entram no histórico apenas depois da entrega, estritamente antes da previsão."""
    ev=base.loc[base.operacional].sort_values('order_delivered_customer_date')
    t=ev.order_delivered_customer_date.to_numpy(dtype='datetime64[ns]')
    q=tabela.momento_predicao.to_numpy(dtype='datetime64[ns]')
    k=np.searchsorted(t,q,side='left')
    soma=np.r_[0,np.cumsum(ev.y.to_numpy())]
    prior=np.where(k>0,(soma[k]+1)/(k+12),.08)
    tabela['taxa_global_passada']=prior
    for col,prefixo in [('seller_id','vendedor'),('rota_uf','rota')]:
        n=np.zeros(len(tabela));s=np.zeros(len(tabela))
        for chave,indices in tabela.groupby(col,sort=False).indices.items():
            grupo=ev.loc[ev[col].eq(chave)]
            tempos=grupo.order_delivered_customer_date.to_numpy(dtype='datetime64[ns]')
            pos=np.searchsorted(tempos,q[indices],side='left')
            n[indices]=pos;s[indices]=np.r_[0,np.cumsum(grupo.y.to_numpy())][pos]
        tabela[prefixo+'_entregas_passadas']=n
        tabela[prefixo+'_taxa_atraso_passada']=(s+20*prior)/(n+20)
    return tabela

ENG_NUM=['prazo_prometido_dias','limite_despacho_dias','margem_limite_ate_prazo',
    'compra_mes','compra_dia_semana','compra_hora','compra_dia_mes','mesma_uf','frete_preco','volume_por_item','peso_por_item',
    'customer_lat','customer_lng','seller_lat','seller_lng']+['log_'+c for c in ['price','freight_value','total_weight','total_volume_cm3','distance_customer_seller']]
ENG_CAT=['grupo_regiao','faixa_distancia','grupo_regiao_distancia','rota_uf','seller_id']
HIST=['taxa_global_passada','vendedor_entregas_passadas','vendedor_taxa_atraso_passada','rota_entregas_passadas','rota_taxa_atraso_passada']
MOVIMENTO=['ja_despachado','tempo_ate_despacho','dias_em_transito','dias_desde_aprovacao','dias_restantes_prazo',
    'fracao_prazo_consumida','distancia_por_dia_restante','despacho_apos_limite']

def construir_contexto(base,nome):
    arquivo=CACHE/f'contexto_{nome}.joblib'
    if arquivo.exists():
        v=joblib.load(arquivo)
        if v['fingerprint']==FINGERPRINT:return v
    d=base.copy() if nome=='atual_retrospectivo' else base.loc[base.operacional].copy()
    if nome in ['atual_retrospectivo','aprovacao']:d['momento_predicao']=d.order_approved_at
    elif nome=='despacho':
        d=d.loc[d.order_delivered_carrier_date.ge(d.order_approved_at) &
                d.order_delivered_carrier_date.lt(d.order_estimated_delivery_date) &
                d.order_delivered_carrier_date.le(d.order_delivered_customer_date)].copy()
        d['momento_predicao']=d.order_delivered_carrier_date
    elif nome.startswith('monitor_'):
        dias=int(nome.split('_')[1]);d['momento_predicao']=d.order_estimated_delivery_date-pd.Timedelta(days=dias)
        # Um pedido já entregue não precisa de intervenção. A posição no tempo é fixada antes do prazo.
        d=d.loc[d.order_approved_at.le(d.momento_predicao) & d.order_delivered_customer_date.gt(d.momento_predicao)].copy()
    else:raise ValueError(nome)
    d=d.sort_values(['momento_predicao','order_id']).reset_index(drop=True)
    d['particao']=np.where(d.momento_predicao.lt(VAL),'treino',np.where(d.momento_predicao.lt(TEST),'validacao','teste'))
    n_antes=d.groupby('particao').size().to_dict()
    if nome!='atual_retrospectivo':
        # Experimento operacional: modelo e limiar só podem usar resultados já disponíveis no corte.
        manter=(d.particao.eq('teste') | (d.particao.eq('treino') & d.order_delivered_customer_date.lt(VAL)) |
                (d.particao.eq('validacao') & d.order_delivered_customer_date.lt(TEST)))
        d=d.loc[manter].reset_index(drop=True)
        d=adicionar_historico(d,base)
        despacho=d.order_delivered_carrier_date
        conhecido=despacho.notna() & despacho.le(d.momento_predicao) & despacho.ge(d.order_approved_at)
        d['ja_despachado']=conhecido.astype(int)
        d['tempo_ate_despacho']=((despacho-d.order_approved_at).dt.total_seconds()/86400).where(conhecido)
        d['dias_em_transito']=((d.momento_predicao-despacho).dt.total_seconds()/86400).where(conhecido)
        d['dias_desde_aprovacao']=(d.momento_predicao-d.order_approved_at).dt.total_seconds()/86400
        d['dias_restantes_prazo']=(d.order_estimated_delivery_date-d.momento_predicao).dt.total_seconds()/86400
        d['fracao_prazo_consumida']=d.dias_desde_aprovacao/d.prazo_prometido_dias.clip(lower=.1)
        d['distancia_por_dia_restante']=d.distance_customer_seller/d.dias_restantes_prazo.clip(lower=.5)
        d['despacho_apos_limite']=(despacho.gt(d.shipping_limit_date)&conhecido).astype(int)
    partes={p:d.loc[d.particao.eq(p)].copy().reset_index(drop=True) for p in ['treino','validacao','teste']}
    assert not set(partes['treino'].order_id)&set(partes['validacao'].order_id)
    assert not set(partes['treino'].order_id)&set(partes['teste'].order_id)
    for p in partes:
        assert len(partes[p]) and set(partes[p].y.unique())=={0,1},(nome,p)
    v={'fingerprint':FINGERPRINT,'nome':nome,'partes':partes,'contagens_antes':n_antes,
       'contagens':{p:{'pedidos':len(x),'atrasos':int(x.y.sum()),'taxa':float(x.y.mean())} for p,x in partes.items()}}
    joblib.dump(v,arquivo,compress=3)
    progresso('Contexto pronto',contexto=nome,contagens=v['contagens'])
    return v

def atributos(nome,versao='engenharia'):
    if nome=='atual_retrospectivo':return ATUAL
    cols=NUM_BASE+CAT_BASE
    if versao!='basico':cols+=ENG_NUM+ENG_CAT
    if versao in ['historico','completo']:cols+=HIST
    if nome not in ['aprovacao','atual_retrospectivo']:cols+=MOVIMENTO
    return list(dict.fromkeys(cols))

def limpar_x(d,cols):
    x=d[cols].copy()
    cat=[c for c in cols if pd.api.types.is_object_dtype(x[c]) or isinstance(x[c].dtype,pd.StringDtype) or isinstance(x[c].dtype,pd.CategoricalDtype)]
    for c in cat:x[c]=x[c].fillna('NAO_INFORMADO').astype(str)
    for c in set(cols)-set(cat):x[c]=pd.to_numeric(x[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    return x,cat

def preprocessador(cols,cat,denso=False):
    num=[c for c in cols if c not in cat]
    return ColumnTransformer([
        ('numericas',Pipeline([('imputar',SimpleImputer(strategy='median',keep_empty_features=True)),('escala',StandardScaler())]),num),
        ('categorias',OneHotEncoder(handle_unknown='infrequent_if_exist',min_frequency=20,max_categories=60,
                                   sparse_output=not denso,dtype=np.float32),cat)],sparse_threshold=0 if denso else 1)

def modelo(spec,cols,cat):
    tipo=spec['tipo'];peso=spec.get('peso',None);par=spec.get('params',{})
    cw='balanced' if peso=='balanced' else ({0:1,1:float(peso)} if peso else None)
    if tipo=='catboost':
        args={'iterations':400,'depth':6,'learning_rate':.05,'l2_leaf_reg':5,'loss_function':'Logloss',
              'random_seed':SEED,'thread_count':THREADS,'verbose':False,'allow_writing_files':False,'cat_features':cat}
        if peso=='balanced':args['auto_class_weights']='Balanced'
        elif peso:args['class_weights']=[1,float(peso)]
        return CatBoostClassifier(**{**args,**par})
    if tipo=='dummy':est=DummyClassifier(strategy='prior')
    elif tipo=='linear':est=LogisticRegression(**{'C':.3,'max_iter':1000,'class_weight':cw,'solver':'liblinear','random_state':SEED,**par})
    elif tipo=='arvore':est=DecisionTreeClassifier(**{'max_depth':6,'min_samples_leaf':40,'class_weight':cw,'random_state':SEED,**par})
    elif tipo in ['random_forest','extra_trees']:
        cls=RandomForestClassifier if tipo=='random_forest' else ExtraTreesClassifier
        est=cls(**{'n_estimators':180,'max_depth':14,'min_samples_leaf':8,'max_features':.7,'n_jobs':THREADS,
                   'class_weight':cw,'random_state':SEED,**par})
    elif tipo=='balanced_rf':est=BalancedRandomForestClassifier(**{'n_estimators':180,'max_depth':12,'min_samples_leaf':5,
        'n_jobs':THREADS,'random_state':SEED,'sampling_strategy':'all','replacement':True,'bootstrap':False,**par})
    elif tipo=='hist_gradient':est=HistGradientBoostingClassifier(**{'max_iter':250,'max_leaf_nodes':15,
        'learning_rate':.05,'min_samples_leaf':40,'l2_regularization':5,'early_stopping':False,'class_weight':cw,
        'random_state':SEED,**par})
    elif tipo=='lightgbm':est=LGBMClassifier(**{'n_estimators':400,'num_leaves':15,'learning_rate':.04,
        'min_child_samples':60,'reg_lambda':5,'colsample_bytree':.9,'class_weight':cw,'n_jobs':THREADS,
        'verbosity':-1,'random_state':SEED,**par})
    elif tipo=='xgboost':est=XGBClassifier(**{'n_estimators':400,'max_depth':4,'learning_rate':.04,
        'min_child_weight':20,'reg_lambda':10,'subsample':.85,'colsample_bytree':.85,'tree_method':'hist',
        'scale_pos_weight':float(peso) if peso and peso!='balanced' else 1,'n_jobs':THREADS,'random_state':SEED,**par})
    else:raise ValueError(tipo)
    prep=preprocessador(cols,cat,denso=tipo=='hist_gradient')
    if spec.get('sampling')=='oversample':
        return ImbPipeline([('preprocessador',prep),('reamostragem',RandomOverSampler(sampling_strategy=.5,random_state=SEED)),('modelo',est)])
    if spec.get('sampling')=='smotenc':
        nums=[c for c in cols if c not in cat]
        numerico=Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]) if spec.get('sampling_scale') else SimpleImputer(strategy='median')
        antes=ColumnTransformer([('num',numerico,nums),
                                ('cat',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),cat)])
        depois=ColumnTransformer([('num',StandardScaler(),list(range(len(nums)))),
            ('cat',OneHotEncoder(handle_unknown='ignore'),list(range(len(nums),len(cols))))])
        return ImbPipeline([('preprocessador',antes),('reamostragem',SMOTENC(categorical_features=list(range(len(nums),len(cols))),
            sampling_strategy=.5,random_state=SEED,k_neighbors=5)),('encoder',depois),('modelo',est)])
    return Pipeline([('preprocessador',prep),('modelo',est)])

def metricas(y,p,limiar=.5):
    y=np.asarray(y,int);p=np.asarray(p,float);pred=p>=limiar
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'pedidos':len(y),'atrasos':int(y.sum()),'taxa_atraso':float(y.mean()),'limiar':float(limiar),
        'f1_macro':f1_score(y,pred,labels=[0,1],average='macro',zero_division=0),
        'f1_micro':f1_score(y,pred,labels=[0,1],average='micro',zero_division=0),
        'precision':precision_score(y,pred,zero_division=0),
        'recall':recall_score(y,pred,zero_division=0),'accuracy':accuracy_score(y,pred),
        'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

def escolher_limiar(y,p):
    y=np.asarray(y,int);p=np.asarray(p,float);ordem=np.argsort(-p,kind='stable');s=p[ordem]
    finais=np.r_[np.flatnonzero(np.diff(s)),len(y)-1]
    tp=np.r_[0,np.cumsum(y[ordem])[finais]];fp=np.r_[0,finais+1]-tp
    fn=y.sum()-tp;tn=len(y)-y.sum()-fp
    lim=np.r_[np.nextafter(s[0],np.inf),s[finais]]
    macro=(2*tp/np.maximum(2*tp+fp+fn,1)+2*tn/np.maximum(2*tn+fp+fn,1))/2
    return float(lim[np.flatnonzero(np.isclose(macro,macro.max()))[0]])

def atualizar_criterio(meta,contexto):
    if meta.get('criterio_selecao')==CRITERIO:return meta
    probs=np.load(CACHE/(meta['id']+'_scores.npz'))
    lim=escolher_limiar(contexto['partes']['validacao'].y,probs['validacao'])
    meta.update(limiar=lim,criterio_selecao=CRITERIO)
    for p in ['treino','validacao']:meta[p]=metricas(contexto['partes'][p].y,probs[p],lim)
    meta['validacao_05']=metricas(contexto['partes']['validacao'].y,probs['validacao'],.5)
    gravar_json(CACHE/(meta['id']+'.json'),meta)
    return meta

def identificador(contexto,spec):
    return contexto['nome']+'__'+spec['nome']

def treinar_candidato(contexto,spec):
    nome=identificador(contexto,spec);cols=spec.get('atributos_explicitos') or atributos(contexto['nome'],spec.get('features','historico'))
    if spec.get('reviews'):cols=cols+['vendedor_nota_passada','vendedor_avaliacoes_passadas']
    if spec.get('sem_historico'):cols=[c for c in cols if c not in HIST]
    if spec.get('sem_ids'):cols=[c for c in cols if c not in ['seller_id','rota_uf']]
    assinatura=hashlib.sha256(json.dumps({'fingerprint':FINGERPRINT,'spec':spec,'cols':cols},sort_keys=True).encode()).hexdigest()
    metapath=CACHE/(nome+'.json');modelpath=CACHE/(nome+'.joblib')
    if metapath.exists() and modelpath.exists():
        meta=json.loads(metapath.read_text(encoding='utf-8'))
        if meta.get('assinatura')==assinatura:return atualizar_criterio(meta,contexto)
    tr=contexto['partes']['treino'];va=contexto['partes']['validacao']
    x,cat=limpar_x(tr,cols);xv,_=limpar_x(va,cols)
    mod=modelo(spec,cols,cat);inicio=time.perf_counter()
    progresso('Treinando',candidato=nome,pedidos=len(tr))
    with threadpool_limits(limits=THREADS):
        mod.fit(x,tr.y)
        pv=mod.predict_proba(xv)[:,list(mod.classes_).index(1)]
        pt=mod.predict_proba(x)[:,list(mod.classes_).index(1)]
    lim=escolher_limiar(va.y,pv)
    meta={'id':nome,'contexto':contexto['nome'],'spec':spec,'assinatura':assinatura,'atributos':cols,
        'cat_features':cat,'limiar':lim,'criterio_selecao':CRITERIO,'segundos_treino':time.perf_counter()-inicio,
        'treino':metricas(tr.y,pt,lim),'validacao':metricas(va.y,pv,lim),'validacao_05':metricas(va.y,pv,.5)}
    joblib.dump(mod,modelpath,compress=3)
    np.savez_compressed(CACHE/(nome+'_scores.npz'),treino=pt,validacao=pv)
    gravar_json(metapath,meta)
    progresso('Candidato concluído',candidato=nome,f1_macro_validacao=round(meta['validacao']['f1_macro'],4),
              segundos=round(meta['segundos_treino'],1))
    return meta

BASE_SPECS=[
    {'nome':'dummy','tipo':'dummy','features':'basico'},
    {'nome':'linear_balanceada','tipo':'linear','peso':'balanced','features':'basico'},
    {'nome':'arvore_balanceada','tipo':'arvore','peso':'balanced'},
    {'nome':'random_forest','tipo':'random_forest','peso':'balanced'},
    {'nome':'extra_trees','tipo':'extra_trees','peso':'balanced'},
    {'nome':'hist_gradient','tipo':'hist_gradient'},
    {'nome':'lightgbm','tipo':'lightgbm'},
    {'nome':'catboost','tipo':'catboost'},
    {'nome':'xgboost','tipo':'xgboost'}]

def executar_benchmark():
    base=preparar_base()
    nomes=['atual_retrospectivo','aprovacao','despacho','monitor_7','monitor_3','monitor_1']
    todos=[]
    for nome in nomes:
        ctx=construir_contexto(base,nome)
        specs=BASE_SPECS if nome in ['aprovacao','despacho','monitor_3'] else [s for s in BASE_SPECS if s['tipo'] in ['dummy','linear','lightgbm','catboost']]
        for spec in specs:todos.append(treinar_candidato(ctx,spec))
    gravar_json(CACHE/'benchmark_validacao.json',todos)
    return todos

In [ ]:
# Evita recuperar modelos de outra configuração ou de outro código por engano.
import nbformat
arquivo_notebook = RAIZ/'src/eda/02_comparacao_modelos_atraso.ipynb'
fontes_codigo = nbformat.read(arquivo_notebook, as_version=4)
codigo_estudo = ''.join(c.source for c in fontes_codigo.cells if c.cell_type=='code' and
                        c.metadata.get('jupyter',{}).get('source_hidden'))
config_estudo = json.loads(json.dumps({'fontes': HASHES,
    'reviews_sha256': sha(RAIZ/'Dados/Brazilian Logistic/olist_order_reviews_dataset.csv'),
    'codigo_sha256': hashlib.sha256(codigo_estudo.encode()).hexdigest(),
    'cortes': CFG['cortes'], 'semente': SEED, 'threads': THREADS,
    'ctgan': GAN_CONFIG, 'limite_ajuste_ctgan': LIMITE_AJUSTE_CTGAN,
    'n_sinteticos': N_SINTETICOS, 'criterio': CRITERIO}))
registro_config = CACHE/'configuracao_estudo.json'
if registro_config.exists():
    assert json.loads(registro_config.read_text(encoding='utf-8')) == config_estudo, (
        'Dados, configuração ou código de treino mudaram. Use outro EXECUCAO para uma nova rodada.')
else:
    gravar_json(registro_config, config_estudo)

## 2. Auditoria e distribuição

As tabelas abaixo mostram os recortes efetivamente utilizados, rótulos sem duração, consistência das datas e quantidade de atrasos. Coordenadas fora da área de referência viram ausentes apenas na cópia experimental; a imputação técnica, quando necessária, é ajustada somente no treino. Nenhum tratamento do usuário é sobrescrito.

In [ ]:
base = preparar_base()
auditoria = json.loads((CACHE/'auditoria_fontes.json').read_text(encoding='utf-8'))
display(pd.DataFrame([auditoria['contagens']]))
contagens = []
for nome in ['atual_retrospectivo','aprovacao','despacho','monitor_7','monitor_3','monitor_1']:
    ctx = construir_contexto(base, nome)
    for p, info in ctx['contagens'].items():
        contagens.append({'contexto': nome, 'particao': p,
                          'antes_restricao_rotulo': ctx['contagens_antes'][p],
                          'fora_por_disponibilidade': ctx['contagens_antes'][p]-info['pedidos'], **info})
contagens = pd.DataFrame(contagens)
display(contagens)
observados = base.loc[base.operacional]
dias_fontes = (observados.order_delivered_customer_date-observados.order_estimated_delivery_date).dt.total_seconds()/86400
assert np.allclose(dias_fontes, observados.delay_in_days, atol=1e-7)
assert np.array_equal(observados.y, observados.delay_in_days.ge(1).astype(int))
assert base.order_id.is_unique
display(base[['delay_in_days','distance_customer_seller','price','qtd_items']].describe(percentiles=[.5,.9,.99]).round(2))
fig, ax = plt.subplots(figsize=(10,4))
taxas = contagens.pivot(index='contexto',columns='particao',values='taxa').reindex(['aprovacao','despacho','monitor_7','monitor_3','monitor_1','atual_retrospectivo'])
taxas[['treino','validacao','teste']].plot.bar(ax=ax,color=['#537b9b','#e1ac4b','#488c68'])
ax.set(ylabel='Proporção de atrasos',xlabel='',title='A população muda com o momento da previsão')
ax.tick_params(axis='x',rotation=25);fig.tight_layout();fig.savefig(FIGURAS/'01_populacoes.png');plt.show()

## 3. Métricas e seleção

Os resultados contêm somente **mae_dias, rmse_dias, r2_dias, mae_dias_atrasados, tn, fn, tp, fp, recall, precision, accuracy, limiar, f1_macro e f1_micro**, além de identificadores e contagens.

| Campo | Interpretação |
|---|---|
| `f1_macro` | média simples do F1 das classes 0 e 1; critério principal |
| `f1_micro` / `accuracy` | coincidem neste problema de uma classe por pedido |
| `precision` | fração dos alertas que realmente atrasaram |
| `recall` | fração dos atrasados que receberam alerta |
| `tn`, `fp`, `fn`, `tp` | no prazo correto, alarme falso, atraso perdido, atraso encontrado |
| `limiar` | corte dos scores escolhido na validação; não é uma garantia de probabilidade calibrada |
| `mae_dias` / `rmse_dias` | erro absoluto médio / erro quadrático médio em dias |
| `r2_dias` | comparação com a média do alvo; pode ser negativo |
| `mae_dias_atrasados` | MAE somente entre pedidos realmente atrasados |
| `particao` | treino, validação ou teste |
| `origem_avaliacao` | real, sintético ou misto |
| `segmento` | todos ou recorte explícito de região/distância; não representa uma nova partição |

O limiar maximiza F1 macro na validação, com desempate pelo maior corte. O classificador é escolhido pelo F1 macro de validação e o regressor pelo menor MAE de validação, separadamente. A comparação final usa os mesmos limiares congelados. Os scores de treino são apenas diagnósticos de ajuste, não estimativas de generalização.

F1 macro de 0,5, isoladamente, é uma exigência modesta em uma base desbalanceada: sempre prever a classe majoritária pode se aproximar desse valor. Por isso a conclusão inclui a referência simples, precision, recall e a matriz de confusão.

In [ ]:
METRICAS_PUBLICAS=['mae_dias','rmse_dias','r2_dias','mae_dias_atrasados',
                  'tn','fn','tp','fp','recall','precision','accuracy','limiar','f1_macro','f1_micro']
REVIEWS=['vendedor_nota_passada','vendedor_avaliacoes_passadas']

def metricas_publicas(y,p,limiar,dias=None,previstos=None):
    y=np.asarray(y,int);pred=np.asarray(p)>=limiar
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    dias=np.full(len(y),np.nan) if dias is None else np.asarray(dias,float)
    previstos=np.full(len(y),np.nan) if previstos is None else np.asarray(previstos,float)
    ok=np.isfinite(dias)&np.isfinite(previstos);at=ok&(y==1)
    return {'mae_dias':mean_absolute_error(dias[ok],previstos[ok]) if ok.any() else np.nan,
        'rmse_dias':float(np.sqrt(mean_squared_error(dias[ok],previstos[ok]))) if ok.any() else np.nan,
        'r2_dias':r2_score(dias[ok],previstos[ok]) if ok.sum()>1 and np.var(dias[ok])>0 else np.nan,
        'mae_dias_atrasados':mean_absolute_error(dias[at],previstos[at]) if at.any() else np.nan,
        'tn':int(tn),'fn':int(fn),'tp':int(tp),'fp':int(fp),
        'recall':recall_score(y,pred,zero_division=0),'precision':precision_score(y,pred,zero_division=0),
        'accuracy':accuracy_score(y,pred),'limiar':float(limiar),
        'f1_macro':f1_score(y,pred,labels=[0,1],average='macro',zero_division=0),
        'f1_micro':f1_score(y,pred,labels=[0,1],average='micro',zero_division=0)}

def contexto_avaliacoes(base,nome):
    """Apenas notas respondidas antes da decisão; a nota do próprio pedido não entra."""
    ctx=construir_contexto(base,nome)
    raw=RAIZ/'Dados/Brazilian Logistic/olist_order_reviews_dataset.csv'
    arquivo=CACHE/f'avaliacoes_{nome}.joblib'
    if arquivo.exists():
        extra=joblib.load(arquivo)
        if extra['fingerprint']==FINGERPRINT and extra['hash_reviews']==sha(raw):
            for p in ctx['partes']:
                for c in REVIEWS:ctx['partes'][p][c]=extra['valores'][p][c]
            return ctx
    ev=pd.read_csv(raw,usecols=['order_id','review_score','review_answer_timestamp'])
    ev['t']=pd.to_datetime(ev.review_answer_timestamp,errors='coerce')
    ev=ev.merge(base[['order_id','seller_id']],on='order_id',validate='many_to_one').dropna(subset=['t','review_score']).sort_values('t')
    valores={}
    for p,d in ctx['partes'].items():
        q=d.momento_predicao.to_numpy(dtype='datetime64[ns]');n=np.zeros(len(d));s=n.copy()
        for key,indices in d.groupby('seller_id',sort=False).indices.items():
            g=ev.loc[ev.seller_id.eq(key)]
            k=np.searchsorted(g.t.to_numpy(dtype='datetime64[ns]'),q[indices],side='left')
            n[indices]=k;s[indices]=np.r_[0,np.cumsum(g.review_score.to_numpy())][k]
            # Reviews antecipadas ou duplicadas do próprio pedido também são removidas.
            respostas=g.groupby('order_id').indices
            for i in indices:
                if d.order_id.iloc[i] in respostas:
                    own=g.iloc[respostas[d.order_id.iloc[i]]]
                    own=own.loc[own.t.lt(d.momento_predicao.iloc[i])]
                    n[i]-=len(own);s[i]-=own.review_score.sum()
        d[REVIEWS[0]]=(s+20*3.5)/(n+20);d[REVIEWS[1]]=n
        valores[p]=d[REVIEWS].copy()
    joblib.dump({'fingerprint':FINGERPRINT,'hash_reviews':sha(raw),'valores':valores},arquivo,compress=3)
    return ctx

def treinar_adicional(ctx,spec):
    return treinar_candidato(ctx,spec)

def executar_expansao():
    base=preparar_base();resultados=[]
    for nome in ['aprovacao','despacho','monitor_3','monitor_7']:
        ctx=contexto_avaliacoes(base,nome)
        specs=[
          {'nome':'linear_engenharia','tipo':'linear','features':'engenharia','peso':'balanced'},
          {'nome':'linear_historico','tipo':'linear','features':'historico','peso':'balanced'},
          {'nome':'catboost_basico','tipo':'catboost','features':'basico'},
          {'nome':'catboost_engenharia','tipo':'catboost','features':'engenharia'},
          {'nome':'lgb_basico','tipo':'lightgbm','features':'basico'},
          {'nome':'lgb_engenharia','tipo':'lightgbm','features':'engenharia'},
          {'nome':'catboost_notas_passadas','tipo':'catboost','reviews':True},
          {'nome':'lgb_notas_passadas','tipo':'lightgbm','reviews':True},
          {'nome':'lgb_peso2','tipo':'lightgbm','peso':2},
          {'nome':'lgb_peso5','tipo':'lightgbm','peso':5},
          {'nome':'catboost_balanceado','tipo':'catboost','peso':'balanced'},
          {'nome':'catboost_regularizado','tipo':'catboost','params':{'depth':4,'iterations':700,'l2_leaf_reg':15}},
          {'nome':'lgb_regularizado','tipo':'lightgbm','params':{'num_leaves':7,'min_child_samples':150,'reg_lambda':20,'n_estimators':650}},
          {'nome':'lgb_profundidade','tipo':'lightgbm','params':{'num_leaves':31,'min_child_samples':100,'n_estimators':600}},
        ]
        if nome in ['aprovacao','despacho']:
            specs += [
              {'nome':'balanced_forest','tipo':'balanced_rf','features':'engenharia'},
              {'nome':'linear_oversample','tipo':'linear','features':'engenharia','sampling':'oversample'},
              {'nome':'lgb_smotenc','tipo':'lightgbm','features':'basico','sampling':'smotenc'},
              {'nome':'lgb_smotenc_padronizado','tipo':'lightgbm','features':'basico','sampling':'smotenc','sampling_scale':True},
              {'nome':'linear_c3','tipo':'linear','features':'historico','peso':'balanced','params':{'C':3.}},
              {'nome':'linear_c003','tipo':'linear','features':'historico','peso':'balanced','params':{'C':.03}},
            ]
        for spec in specs:resultados.append(treinar_adicional(ctx,spec))
    gravar_json(CACHE/'expansao_validacao.json',resultados)
    return resultados

def prever_pacote(pacote,d):
    """Roteia para estimador global, especialista ou ensemble; não consulta rótulos."""
    if pacote['tipo']=='ensemble':
        return sum(w*prever_estimador(mid,d) for mid,w in zip(pacote['modelos'],pacote['pesos']))
    x,_=limpar_x(d,pacote['atributos'])
    p=pacote['global'].predict_proba(x)[:,1]
    for grupo,mod in pacote.get('especialistas',{}).items():
        ix=d[pacote['grupo']].eq(grupo).to_numpy()
        if ix.any():p[ix]=mod.predict_proba(x.loc[ix])[:,1]
    return p

def prever_estimador(mid,d):
    meta=json.loads((CACHE/(mid+'.json')).read_text(encoding='utf-8'))
    mod=joblib.load(CACHE/(mid+'.joblib'))
    with threadpool_limits(limits=THREADS):
        if isinstance(mod,dict):return prever_pacote(mod,d)
        x,_=limpar_x(d,meta['atributos'])
        return mod.predict_proba(x)[:,list(mod.classes_).index(1)]

def guardar_especial(meta,mod,ctx):
    mid=meta['id'];probs={p:prever_pacote(mod,ctx['partes'][p]) for p in ['treino','validacao']}
    lim=escolher_limiar(ctx['partes']['validacao'].y,probs['validacao'])
    meta.update(limiar=lim,treino=metricas(ctx['partes']['treino'].y,probs['treino'],lim),
        validacao=metricas(ctx['partes']['validacao'].y,probs['validacao'],lim),
        validacao_05=metricas(ctx['partes']['validacao'].y,probs['validacao'],.5),fingerprint=FINGERPRINT,criterio_selecao=CRITERIO)
    joblib.dump(mod,CACHE/(mid+'.joblib'),compress=3)
    np.savez_compressed(CACHE/(mid+'_scores.npz'),**probs);gravar_json(CACHE/(mid+'.json'),meta)
    progresso('Estratégia concluída',candidato=mid,f1_macro_validacao=meta['validacao']['f1_macro'])
    return meta

def executar_regioes():
    base=preparar_base();res=[]
    for nome in ['aprovacao','despacho','monitor_3']:
        ctx=contexto_avaliacoes(base,nome);tr=ctx['partes']['treino']
        for grupo in ['grupo_regiao','grupo_regiao_distancia']:
            mid=nome+'__especialistas_'+grupo
            if (CACHE/(mid+'.json')).exists():
                res.append(json.loads((CACHE/(mid+'.json')).read_text(encoding='utf-8')));continue
            cols=atributos(nome,'historico');x,cat=limpar_x(tr,cols);inicio=time.perf_counter()
            spec={'nome':mid.split('__')[1],'tipo':'lightgbm','params':{'num_leaves':7,'min_child_samples':80,'n_estimators':300}}
            global_=modelo(spec,cols,cat);especialistas={};contagens={}
            with threadpool_limits(limits=THREADS):
                global_.fit(x,tr.y)
                for g,ix in tr.groupby(grupo,sort=True).indices.items():
                    y=tr.y.iloc[ix];contagens[g]={'pedidos':len(ix),'atrasos':int(y.sum()),'especialista':False}
                    if len(ix)<500 or min(int(y.sum()),int((1-y).sum()))<30:continue
                    local=modelo(spec,cols,cat);local.fit(x.iloc[ix],y);especialistas[g]=local;contagens[g]['especialista']=True
            pacote={'tipo':'regioes','grupo':grupo,'global':global_,'especialistas':especialistas,'atributos':cols}
            meta={'id':mid,'contexto':nome,'spec':spec,'atributos':cols,'cat_features':cat,'grupos':contagens,
                  'segundos_treino':time.perf_counter()-inicio}
            res.append(guardar_especial(meta,pacote,ctx))
    gravar_json(CACHE/'regioes_validacao.json',res);return res

def candidatos():
    saida=[]
    for p in sorted(CACHE.glob('*.json')):
        try:m=json.loads(p.read_text(encoding='utf-8'))
        except (ValueError,OSError):continue
        if isinstance(m,dict) and 'id' in m and 'spec' in m and 'validacao' in m:saida.append(m)
    return saida

def executar_ensembles():
    base=preparar_base();res=[]
    for nome in ['aprovacao','despacho','monitor_7','monitor_3','monitor_1']:
        ctx=contexto_avaliacoes(base,nome)
        lista=sorted([m for m in candidatos() if m['contexto']==nome and m['spec']['tipo']!='dummy' and
                      not m['spec'].get('ensemble') and not m['spec'].get('sinteticos')],key=lambda m:-m['validacao']['f1_macro'])[:3]
        for pesos in [[1/3]*3,[.6,.2,.2]]:
            mid=nome+'__voto_'+('igual' if pesos[0]<.5 else 'ponderado')
            if (CACHE/(mid+'.json')).exists():res.append(json.loads((CACHE/(mid+'.json')).read_text(encoding='utf-8')));continue
            mod={'tipo':'ensemble','modelos':[m['id'] for m in lista],'pesos':pesos,'atributos':lista[0]['atributos']}
            meta={'id':mid,'contexto':nome,'spec':{'nome':mid.split('__')[1],'tipo':'lightgbm','ensemble':True},
                  'componentes':mod['modelos'],'pesos':pesos,'atributos':lista[0]['atributos'],
                  'cat_features':lista[0]['cat_features'],'segundos_treino':0.}
            res.append(guardar_especial(meta,mod,ctx))
    gravar_json(CACHE/'ensembles_validacao.json',res);return res

## 4. Modelos e pipelines comparados

São comparados Dummy, regressão logística, árvore, Random Forest, Extra Trees, HistGradientBoosting, LightGBM, CatBoost, XGBoost e Balanced Random Forest, além de ensembles e especialistas regionais.

- **Pipeline básico:** atributos disponíveis, imputação de ausências técnicas no treino, padronização numérica e OneHotEncoder. CatBoost trata as categorias de forma nativa.
- **Engenharia:** prazos prometido/de despacho, calendário, coordenadas, relações de frete/peso/volume, regiões, rota e faixas de distância.
- **Histórico:** entregas e atrasos anteriores de vendedor/rota; comparação adicional com notas anteriores do vendedor.
- **Desbalanceamento:** pesos de classe, RandomOverSampler, SMOTENC e Balanced Random Forest. Reamostragem acontece somente no `fit` do treino.
- **Regiões:** SP separado, restante do Sudeste, Sul, Nordeste, Norte, Centro-Oeste; também região × distância de 500 km. Especialistas exigem ao menos 500 pedidos e 30 exemplos de cada classe; os demais usam o modelo global. O limiar permanece único por candidato.
- **Ensemble:** média dos scores dos três melhores candidatos elegíveis de validação, com pesos iguais ou 0,6/0,2/0,2.

As grades são finitas e estão no código: o estudo compara essas alternativas, sem afirmar que esgotou todos os algoritmos/hiperparâmetros possíveis. Mais atributos ou um pipeline mais elaborado não garantem melhora; a comparação usa a mesma população dentro de cada contexto.

In [ ]:
iniciais = executar_benchmark()
expansao = executar_expansao()
regionais = executar_regioes()
ensembles = executar_ensembles()
validacao_modelos = pd.DataFrame([{'contexto':m['contexto'],'modelo':m['id'].split('__')[1],
    **{k:m['validacao'][k] for k in ['f1_macro','f1_micro','precision','recall','accuracy','limiar']}}
    for m in candidatos()])
display(validacao_modelos.sort_values(['contexto','f1_macro'],ascending=[True,False]).groupby('contexto',sort=False).head(5))

## 5. CTGAN e comparação real/sintético/misto

A CTGAN recebe somente uma amostra reproduzível de até 20 mil pedidos de treino real e 12 atributos operacionais, além de `delay_in_days`; treina por 30 épocas. A classe é derivada de `delay_in_days >= 1`, mantendo coerência do alvo. Contagens e grandezas físicas negativas geradas são corrigidas conforme registrado; os dados reais não mudam.

Geram-se pools independentes de 10 mil/3 mil/3 mil amostras para treino/validação/teste, com IDs artificiais e remoção de duplicatas entre pools. Há CSVs reais, sintéticos e mistos nas três partições. Três famílias de classificadores usam treino real, sintético ou misto com os mesmos atributos. A escolha primária e o limiar usam **validação real comum**; todos são pontuados depois nas nove combinações de partição e origem. Métricas em sintéticos são diagnósticos do gerador, não validação independente da operação real.

O limite de amostra/épocas é um orçamento explícito deste estudo; o resultado não demonstra que qualquer configuração de CTGAN teria o mesmo comportamento.

In [ ]:
COLS_GAN=NUM_BASE+CAT_BASE+['prazo_prometido_dias','limite_despacho_dias']
PASTA_CONJUNTOS=RAIZ/'Dados'/EXECUCAO.replace('modelos','conjuntos')
GAN_CONFIG=globals().get('GAN_CONFIG',{'epochs':30,'batch_size':500,'pac':10,'generator_dim':(128,128),
            'discriminator_dim':(128,128),'verbose':True,'enable_gpu':False})
LIMITE_AJUSTE_CTGAN=globals().get('LIMITE_AJUSTE_CTGAN',20000)
N_SINTETICOS=globals().get('N_SINTETICOS',{'treino':10000,'validacao':3000,'teste':3000})

def preparar_ctgan(ctx):
    from ctgan import CTGAN
    import torch
    torch.set_num_threads(THREADS);torch.manual_seed(SEED)
    fonte=ctx['partes']['treino'];tabela=fonte[COLS_GAN+['delay_in_days']].copy()
    # Limite explícito de custo do experimento. Amostra apenas de treino, sem balanceamento.
    tabela=tabela.sample(n=min(LIMITE_AJUSTE_CTGAN,len(tabela)),random_state=SEED)
    discretas=CAT_BASE+['qtd_items']
    for c in discretas:tabela[c]=tabela[c].astype(str)
    for c in set(tabela)-set(discretas):tabela[c]=tabela[c].fillna(tabela[c].median()).astype(float)
    path=CACHE/'ctgan_gerador.joblib';assinatura=hashlib.sha256(json.dumps({'fonte':FINGERPRINT,'config':GAN_CONFIG,'cols':COLS_GAN,'n':len(tabela)},sort_keys=True).encode()).hexdigest()
    if path.exists():
        pkg=joblib.load(path)
        if pkg['assinatura']==assinatura:return pkg
    progresso('Treinando CTGAN',pedidos=len(tabela),epocas=GAN_CONFIG['epochs'])
    inicio=time.perf_counter();gan=CTGAN(**GAN_CONFIG);gan.set_random_state(SEED)
    with threadpool_limits(limits=THREADS):gan.fit(tabela,discrete_columns=discretas)
    pkg={'gerador':gan,'assinatura':assinatura,'pedidos_ajuste':len(tabela),'config':GAN_CONFIG,'segundos':time.perf_counter()-inicio}
    joblib.dump(pkg,path,compress=3);progresso('CTGAN ajustada',segundos=pkg['segundos'])
    return pkg

def gerar_conjuntos(ctx):
    arquivo=CACHE/'ctgan_conjuntos.joblib'
    if arquivo.exists():
        antigo=joblib.load(arquivo)
        config_antiga=json.loads(json.dumps(antigo.get('config',{})))
        config_atual=json.loads(json.dumps(GAN_CONFIG))
        quantidades={p:len(antigo['conjuntos'][p]['sintetico']) for p in N_SINTETICOS}
        if (antigo['fingerprint']==FINGERPRINT and config_antiga==config_atual and
            antigo.get('pedidos_ajuste')==min(LIMITE_AJUSTE_CTGAN,len(ctx['partes']['treino'])) and quantidades==N_SINTETICOS):return antigo
    pkg=preparar_ctgan(ctx);PASTA_CONJUNTOS.mkdir(parents=True,exist_ok=True)
    conjuntos={};vistos=set();relatorio={}
    for i,p in enumerate(['treino','validacao','teste']):
        n=N_SINTETICOS[p]
        pkg['gerador'].set_random_state(SEED+1000+i)
        with threadpool_limits(limits=THREADS):s=pkg['gerador'].sample(n)
        for c in set(COLS_GAN+['delay_in_days'])-set(CAT_BASE):s[c]=pd.to_numeric(s[c],errors='raise')
        negativos={c:int(s[c].lt(0).sum()) for c in NUM_BASE if c!='approval_date_in_days'}
        for c in NUM_BASE:s[c]=s[c].clip(lower=0)
        s['qtd_items']=s.qtd_items.round().clip(lower=1)
        s['prazo_prometido_dias']=s.prazo_prometido_dias.clip(lower=.01)
        # A classe deriva da duração, preservando o alvo >=1 dia; não é gerada à parte.
        s['y']=s.delay_in_days.ge(1).astype(int);s['delay_status']=s.y
        hashes=pd.util.hash_pandas_object(s[COLS_GAN+['delay_in_days']],index=False)
        duplicados=hashes.isin(vistos)|hashes.duplicated();s=s.loc[~duplicados].reset_index(drop=True)
        vistos.update(hashes.loc[~duplicados]);s['order_id']=[f'EDA-SYN-{p}-{j:06d}' for j in range(len(s))]
        s['particao']=p;s['origem_dado']='sintetico_ctgan';s['delay_status_source']='ctgan'
        s['grupo_regiao']=s.customer_state.map(REGIOES).fillna('Nao_informada')
        s['faixa_distancia']=np.where(s.distance_customer_seller.gt(500),'acima_500','ate_500')
        s['grupo_regiao_distancia']=s.grupo_regiao+'__'+s.faixa_distancia
        r=ctx['partes'][p].copy();r['origem_dado']='real_olist'
        misto=pd.concat([r,s],ignore_index=True)
        conjuntos[p]={'real':r,'sintetico':s,'misto':misto}
        relatorio[p]={'pedidos':len(s),'atrasos':int(s.y.sum()),'duplicados_removidos':int(duplicados.sum()),'negativos_antes_ajuste':negativos}
        for modo,d in conjuntos[p].items():
            d[['order_id','particao','origem_dado']+COLS_GAN+['delay_in_days','delay_status']].to_csv(PASTA_CONJUNTOS/f'{modo}_{p}.csv',index=False)
    v={'fingerprint':FINGERPRINT,'conjuntos':conjuntos,'relatorio':relatorio,'config':GAN_CONFIG,'pedidos_ajuste':pkg['pedidos_ajuste']}
    joblib.dump(v,arquivo,compress=3);gravar_json(CACHE/'ctgan_geracao.json',{k:v[k] for k in ['fingerprint','relatorio','config','pedidos_ajuste']})
    return v

def executar_sinteticos():
    base=preparar_base();ctx=construir_contexto(base,'aprovacao');pools=gerar_conjuntos(ctx);res=[]
    for modo in ['real','sintetico','misto']:
        # A seleção primária sempre usa a mesma validação REAL.
        partes={**ctx['partes'],'treino':pools['conjuntos']['treino'][modo]}
        c={**ctx,'partes':partes}
        for tipo in ['linear','lightgbm','catboost']:
            spec={'nome':f'ctgan_{modo}_{tipo}','tipo':tipo,'atributos_explicitos':COLS_GAN,'sinteticos':True,'modo_treino':modo}
            if tipo=='linear':spec['peso']='balanced'
            res.append(treinar_candidato(c,spec))
    gravar_json(CACHE/'ctgan_validacao.json',res);return res

In [ ]:
sinteticos = executar_sinteticos()
geracao = json.loads((CACHE/'ctgan_geracao.json').read_text(encoding='utf-8'))
display(pd.DataFrame(geracao['relatorio']).T)
display(pd.DataFrame([{'modelo':m['id'].split('__')[1], **{k:m['validacao'][k] for k in
    ['f1_macro','f1_micro','precision','recall','accuracy','limiar']}} for m in sinteticos]))
print('Conjuntos para inspeção:', PASTA_CONJUNTOS)

## 6. Previsão em dias, estabilidade temporal e congelamento

Cada candidato é acompanhado por um regressor da família correspondente, com os mesmos atributos. Regressão logística usa Ridge para dias; Dummy usa a mediana; Balanced Random Forest usa RandomForestRegressor; especialistas/ensembles de classificação são acompanhados por um regressor global explicitamente identificado. Regressores idênticos são reutilizados. Pesos e reamostragem de classificação não são aplicados ao alvo contínuo.

Não se força a previsão de dias a concordar com o classificador: são objetivos distintos. O pacote final pode combinar o melhor classificador com outro regressor de menor MAE. As linhas do CSV mostram o regressor associado a cada candidato; o manifesto registra a combinação selecionada.

Dois testes retrospectivos adicionais, anteriores a março de 2018, verificam a estabilidade temporal de pipelines representativos com limiar fixo 0,5. Eles não usam o teste final e não reajustam o corte no período avaliado.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

CSV_RESULTADOS=RAIZ/'Dados/resultados_modelos'/f'{EXECUCAO}.csv'
MODELOS_FINAIS=RAIZ/'Modelos'/EXECUCAO
CONTEXTOS_MEM={}

def partes_modelo(meta,base):
    nome=meta['contexto']
    if nome not in CONTEXTOS_MEM:
        CONTEXTOS_MEM[nome]=contexto_avaliacoes(base,nome) if nome!='atual_retrospectivo' else construir_contexto(base,nome)
    ctx=CONTEXTOS_MEM[nome]
    if meta['spec'].get('sinteticos'):
        pools=gerar_conjuntos(ctx)
        return {**ctx['partes'],'treino':pools['conjuntos']['treino'][meta['spec']['modo_treino']]}
    return ctx['partes']

def criar_regressor(meta,cols,cat):
    tipo=meta['spec']['tipo'];par=meta['spec'].get('params',{})
    if tipo=='catboost':
        return CatBoostRegressor(**{'iterations':400,'depth':6,'learning_rate':.05,'l2_leaf_reg':5,
            'loss_function':'RMSE','random_seed':SEED,'thread_count':THREADS,'verbose':False,
            'allow_writing_files':False,'cat_features':cat,**par})
    if tipo=='dummy':reg=DummyRegressor(strategy='median')
    elif tipo=='linear':reg=Ridge(alpha=10.)
    elif tipo=='arvore':reg=DecisionTreeRegressor(max_depth=6,min_samples_leaf=40,random_state=SEED)
    elif tipo in ['random_forest','balanced_rf','extra_trees']:
        cls=ExtraTreesRegressor if tipo=='extra_trees' else RandomForestRegressor
        reg=cls(n_estimators=180,max_depth=14,min_samples_leaf=8,max_features=.7,n_jobs=THREADS,random_state=SEED)
    elif tipo=='hist_gradient':reg=HistGradientBoostingRegressor(max_iter=250,max_leaf_nodes=15,learning_rate=.05,min_samples_leaf=40,l2_regularization=5,early_stopping=False,random_state=SEED)
    elif tipo=='xgboost':reg=XGBRegressor(**{'n_estimators':400,'max_depth':4,'learning_rate':.04,'min_child_weight':20,'reg_lambda':10,'subsample':.85,'colsample_bytree':.85,'tree_method':'hist','n_jobs':THREADS,'random_state':SEED,**par})
    else:reg=LGBMRegressor(**{'n_estimators':400,'num_leaves':15,'learning_rate':.04,'min_child_samples':60,'reg_lambda':5,'colsample_bytree':.9,'verbosity':-1,'n_jobs':THREADS,'random_state':SEED,**par})
    return Pipeline([('preprocessador',preprocessador(cols,cat,denso=tipo=='hist_gradient')),('modelo',reg)])

def treinar_regressao(meta,partes):
    cols=meta['atributos'];tr=partes['treino'];va=partes['validacao']
    x,cat=limpar_x(tr,cols);xv,_=limpar_x(va,cols);reg=criar_regressor(meta,cols,cat)
    parametros=str(reg.get_params(deep=True));ids=hashlib.sha256('\n'.join(tr.order_id).encode()).hexdigest()
    assinatura=hashlib.sha256((FINGERPRINT+meta['contexto']+ids+str(cols)+parametros).encode()).hexdigest()
    chave='reg_'+assinatura[:20];path=CACHE/(chave+'.json')
    if path.exists():return json.loads(path.read_text(encoding='utf-8'))
    ok=np.isfinite(tr.delay_in_days);inicio=time.perf_counter()
    progresso('Treinando previsão em dias',candidato=meta['id'])
    with threadpool_limits(limits=THREADS):
        reg.fit(x.loc[ok],tr.loc[ok,'delay_in_days'])
        pt=reg.predict(x);pv=reg.predict(xv)
    m=metricas_publicas(va.y,np.zeros(len(va)),.5,va.delay_in_days,pv)
    info={'id':chave,'assinatura':assinatura,'tipo':type(reg[-1]).__name__ if isinstance(reg,Pipeline) else type(reg).__name__,
          'atributos':cols,'cat_features':cat,'pedidos_treino':int(ok.sum()),'segundos_treino':time.perf_counter()-inicio,
          'validacao':{k:m[k] for k in METRICAS_PUBLICAS[:4]}}
    joblib.dump(reg,CACHE/(chave+'.joblib'),compress=3)
    np.savez_compressed(CACHE/(chave+'_scores.npz'),treino=pt,validacao=pv)
    gravar_json(path,info);return info

def executar_regressoes():
    base=preparar_base();registro=[]
    for meta in candidatos():
        partes=partes_modelo(meta,base);reg=treinar_regressao(meta,partes)
        registro.append({'modelo':meta['id'],'regressor':reg['id'],'validacao':reg['validacao']})
    gravar_json(CACHE/'regressoes_validacao.json',registro);return registro

def congelar_selecao():
    path=CACHE/'selecao_congelada.json'
    lista=candidatos();registros=json.loads((CACHE/'regressoes_validacao.json').read_text(encoding='utf-8'))
    reg={r['modelo']:r for r in registros}
    assert all(m['id'] in reg for m in lista)
    escolhidos={}
    for nome in sorted({m['contexto'] for m in lista}):
        pool=[m for m in lista if m['contexto']==nome]
        cls=sorted(pool,key=lambda m:(-m['validacao']['f1_macro'],-m['validacao']['precision'],m['id']))[0]
        regmeta=min(pool,key=lambda m:(reg[m['id']]['validacao']['mae_dias'],m['id']))
        escolhidos[nome]={'classificador':cls['id'],'regressor':reg[regmeta['id']]['regressor'],
            'modelo_origem_regressor':regmeta['id'],'limiar':cls['limiar'],'f1_macro_validacao':cls['validacao']['f1_macro']}
    manifest={'fingerprint':FINGERPRINT,'criterio':'f1_macro_validacao','escolhidos':escolhidos,
        'candidatos':[{'id':m['id'],'spec':m['spec'],'limiar':m['limiar'],'atributos':m['atributos'],
                      'modelo_sha256':sha(CACHE/(m['id']+'.joblib')),'regressor':reg[m['id']]['regressor']} for m in lista]}
    assinatura=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
    if path.exists():
        velho=json.loads(path.read_text(encoding='utf-8'))
        if velho['assinatura']!=assinatura:
            raise ValueError('A seleção já foi congelada. Para novos experimentos use outra execução/cache, preservando este teste.')
        return velho
    manifest.update(assinatura=assinatura,congelado_utc=datetime.now(timezone.utc).isoformat())
    gravar_json(path,manifest);progresso('Seleção congelada antes do teste',escolhidos=escolhidos)
    return manifest

def prever_dias(registro,d):
    reg=joblib.load(CACHE/(registro['id']+'.joblib'));x,_=limpar_x(d,registro['atributos'])
    with threadpool_limits(limits=THREADS):return reg.predict(x)

def linha_resultado(meta,reg,particao,origem,segmento,d,pred,pr,manifest):
    resultado=metricas_publicas(d.y,pred,meta['limiar'],d.delay_in_days,pr)
    es=manifest['escolhidos'][meta['contexto']]
    return {'execucao':EXECUCAO,'contexto':meta['contexto'],'modelo':meta['id'].split('__',1)[1],
        'modelo_dias':reg['tipo'],'id_modelo_dias':reg['id'],'cenario_treino':meta['spec'].get('modo_treino','real'),
        'particao':particao,'segmento':segmento,'origem_avaliacao':origem,'pedidos':len(d),
        'pedidos_atrasados':int(d.y.sum()),'pedidos_avaliados_regressao':int(np.isfinite(d.delay_in_days).sum()),
        'rotulos_pln':int(d.get('delay_status_source',pd.Series('',index=d.index)).eq('pln').sum()),
        'selecionado_classificador':es['classificador']==meta['id'],
        'selecionado_regressor':es['regressor']==reg['id'],
        'criterio_selecao':'f1_macro_validacao','parametros':json.dumps(meta['spec'],ensure_ascii=False,sort_keys=True),**resultado}

def avaliar_final():
    manifest=congelar_selecao();base=preparar_base();linhas=[]
    lista={m['id']:m for m in candidatos()}
    for item in manifest['candidatos']:
        mid=item['id'];meta=lista[mid]
        assert sha(CACHE/(mid+'.joblib'))==item['modelo_sha256']
        partes=partes_modelo(meta,base);reg=json.loads((CACHE/(item['regressor']+'.json')).read_text(encoding='utf-8'))
        scores=np.load(CACHE/(mid+'_scores.npz'));dias=np.load(CACHE/(reg['id']+'_scores.npz'))
        for p,d in partes.items():
            cacheteste=CACHE/(mid+'_teste.npz')
            if p=='teste':
                if cacheteste.exists():
                    z=np.load(cacheteste);pc=z['score'];pr=z['dias']
                    assert np.array_equal(z['order_id'],d.order_id.to_numpy(dtype=str))
                else:
                    pc=prever_estimador(mid,d);pr=prever_dias(reg,d)
                    np.savez_compressed(cacheteste,order_id=d.order_id.to_numpy(dtype=str),score=pc,dias=pr)
            else:pc=scores[p];pr=dias[p]
            origem=meta['spec'].get('modo_treino','real') if p=='treino' else 'real'
            linhas.append(linha_resultado(meta,reg,p,origem,'todos',d,pc,pr,manifest))
            # Detalhamento geográfico dos selecionados; nenhuma escolha é feita por esse teste.
            if manifest['escolhidos'][meta['contexto']]['classificador']==mid:
                for coluna in ['grupo_regiao','faixa_distancia','grupo_regiao_distancia']:
                    for nome,ix in d.groupby(coluna,sort=True).indices.items():
                        linhas.append(linha_resultado(meta,reg,p,origem,coluna+':'+str(nome),d.iloc[ix],pc[ix],pr[ix],manifest))
        if meta['spec'].get('sinteticos'):
            ctx=CONTEXTOS_MEM[meta['contexto']];pools=gerar_conjuntos(ctx)
            for p in ['treino','validacao','teste']:
                for origem in ['real','sintetico','misto']:
                    if (p=='treino' and origem==meta['spec']['modo_treino']) or (p!='treino' and origem=='real'):continue
                    d=pools['conjuntos'][p][origem];pc=prever_estimador(mid,d);pr=prever_dias(reg,d)
                    linhas.append(linha_resultado(meta,reg,p,origem,'todos',d,pc,pr,manifest))
        progresso('Avaliado',candidato=mid)
    tabela=pd.DataFrame(linhas)
    # Ordenação fixa facilita comparação entre execuções e não introduz outras métricas.
    infos=[c for c in tabela if c not in METRICAS_PUBLICAS]
    tabela=tabela[infos+METRICAS_PUBLICAS].sort_values(['contexto','modelo','particao','origem_avaliacao','segmento'])
    assert not tabela.duplicated(['execucao','contexto','modelo','particao','origem_avaliacao','segmento']).any()
    assert np.allclose(tabela.f1_micro,tabela.accuracy)
    assert (tabela[['tn','fn','tp','fp']].sum(axis=1)==tabela.pedidos).all()
    assert all(sha(RAIZ/p)==h for p,h in HASHES.items()),'Uma fonte foi alterada durante o estudo.'
    CSV_RESULTADOS.parent.mkdir(parents=True,exist_ok=True);temporario=CSV_RESULTADOS.with_suffix('.csv.tmp')
    tabela.to_csv(temporario,index=False);temporario.replace(CSV_RESULTADOS)
    MODELOS_FINAIS.mkdir(parents=True,exist_ok=True)
    # Pacotes completos de classificação e dias, escolhidos independentemente na validação.
    for nome,es in manifest['escolhidos'].items():
        meta=lista[es['classificador']];mod=joblib.load(CACHE/(meta['id']+'.joblib'))
        if isinstance(mod,dict) and mod['tipo']=='ensemble':
            def carregar_componente(mid):
                v=joblib.load(CACHE/(mid+'.joblib'))
                return {'modelo':v,'metadados':lista[mid]}
            mod={**mod,'componentes_embutidos':[carregar_componente(mid) for mid in mod['modelos']]}
        regmeta=json.loads((CACHE/(es['regressor']+'.json')).read_text(encoding='utf-8'))
        pacote={'classificador':mod,'regressor':joblib.load(CACHE/(es['regressor']+'.joblib')),
            'limiar':meta['limiar'],'atributos_classificador':meta['atributos'],
            'atributos_regressor':regmeta['atributos'],'contexto':nome,'criterio_selecao':manifest['criterio'],
            'fingerprint':FINGERPRINT,'protocolo':'Exige os mesmos atributos calculados no momento da decisão; não é o contrato da API antiga.'}
        joblib.dump(pacote,MODELOS_FINAIS/(nome+'.joblib'),compress=3)
    gravar_json(MODELOS_FINAIS/'selecao.json',manifest)
    progresso('CSV de avaliação concluído',csv=str(CSV_RESULTADOS),linhas=len(tabela),candidatos=len(lista))
    return tabela

def backtest_temporal():
    """Duas janelas anteriores a março; teste final permanece fora do ajuste."""
    path=CACHE/'backtest_temporal.json'
    if path.exists():return pd.DataFrame(json.loads(path.read_text(encoding='utf-8')))
    base=preparar_base();ctx=contexto_avaliacoes(base,'aprovacao')
    dados=pd.concat(ctx['partes'].values(),ignore_index=True)
    resultados=[]
    specs=[{'nome':'linear_basico','tipo':'linear','features':'basico','peso':'balanced'},
           {'nome':'linear_engenharia','tipo':'linear','features':'engenharia','peso':'balanced'},
           {'nome':'lgb_basico','tipo':'lightgbm','features':'basico'}]
    for corte,fim in [('2017-10-01','2017-12-01'),('2018-01-01','2018-03-01')]:
        corte=pd.Timestamp(corte);fim=pd.Timestamp(fim)
        treino=dados.loc[dados.momento_predicao.lt(corte)&dados.order_delivered_customer_date.lt(corte)]
        val=dados.loc[dados.momento_predicao.ge(corte)&dados.momento_predicao.lt(fim)&dados.order_delivered_customer_date.lt(fim)]
        for spec in specs:
            cols=atributos('aprovacao',spec['features']);x,cat=limpar_x(treino,cols);xv,_=limpar_x(val,cols)
            mod=modelo(spec,cols,cat)
            with threadpool_limits(limits=THREADS):mod.fit(x,treino.y);pv=mod.predict_proba(xv)[:,1]
            # Fixo em 0,5: mede estabilidade de janelas sem otimizar no período pontuado.
            resultados.append({'modelo':spec['nome'],'inicio_validacao':str(corte.date()),'fim_validacao':str(fim.date()),
                'pedidos_treino':len(treino),'pedidos_validacao':len(val),**metricas_publicas(val.y,pv,.5)})
    gravar_json(path,resultados);return pd.DataFrame(resultados)

In [ ]:
regressoes = executar_regressoes()
backtests = backtest_temporal()
display(backtests.drop(columns=['mae_dias','rmse_dias','r2_dias','mae_dias_atrasados']))
selecao = congelar_selecao()
display(pd.DataFrame(selecao['escolhidos']).T)
print('Seleção congelada em UTC:', selecao['congelado_utc'])

## 7. Teste final e resultados

Somente a partir desta célula são pontuados os pedidos de teste. A seleção acima já está salva; o teste não troca o vencedor. Todos os candidatos predefinidos são exportados para permitir a inspeção pedida. Resultados de contextos diferentes devem ser comparados junto com momento da decisão, população e quantidade de pedidos.

In [ ]:
resultados = avaliar_final()
finalistas = resultados.loc[resultados.particao.eq('teste') & resultados.segmento.eq('todos') &
    resultados.origem_avaliacao.eq('real') & resultados.selecionado_classificador].copy()
display(finalistas[['contexto','modelo','cenario_treino','pedidos']+METRICAS_PUBLICAS].round(4))
print('CSV único de resultados:', CSV_RESULTADOS)
print('Pacotes selecionados:', MODELOS_FINAIS)

## 8. Comparações e diagnósticos

Os gráficos abaixo usam os vencedores escolhidos na validação, mesmo quando outro candidato obteve um resultado de teste numericamente maior. Nenhum candidato é promovido depois de examinar o teste. A linha de 0,5 representa a meta de F1 macro; a referência Dummy ajuda a interpretar o ganho.

In [ ]:
todos = resultados.loc[resultados.segmento.eq('todos') & resultados.origem_avaliacao.eq('real')].copy()
ordem_contextos = ['aprovacao','despacho','monitor_7','monitor_3','monitor_1','atual_retrospectivo']
fig, ax = plt.subplots(figsize=(11,4.5))
selecionados = todos.loc[todos.selecionado_classificador & todos.particao.isin(['validacao','teste'])]
graf = selecionados.pivot(index='contexto',columns='particao',values='f1_macro').reindex(ordem_contextos)
graf[['validacao','teste']].plot.bar(ax=ax,color=['#8da6bd','#32775a'])
dummy = todos.loc[todos.modelo.eq('dummy') & todos.particao.eq('teste')].set_index('contexto').reindex(ordem_contextos)
ax.plot(np.arange(len(ordem_contextos)),dummy.f1_macro,'o--',color='#a15a38',label='Dummy no teste')
ax.axhline(.5,color='#333333',linestyle=':',label='Meta F1 macro = 0,5')
ax.set(ylim=(0,1),ylabel='F1 macro',xlabel='',title='Modelos escolhidos na validação; teste real')
ax.tick_params(axis='x',rotation=25);ax.legend(loc='upper left',ncol=2,fontsize=8)
fig.tight_layout();fig.savefig(FIGURAS/'02_modelos_selecionados.png');plt.show()

pares = ['linear_balanceada','linear_engenharia','linear_historico','linear_oversample',
         'catboost_basico','catboost_engenharia','catboost','catboost_notas_passadas',
         'ctgan_real_linear','ctgan_real_catboost','especialistas_grupo_regiao',
         'especialistas_grupo_regiao_distancia']
comparacao_pipeline = todos.loc[todos.contexto.eq('aprovacao') & todos.modelo.isin(pares) &
    todos.particao.isin(['validacao','teste']),['modelo','particao','f1_macro','precision','recall','limiar']]
display(comparacao_pipeline.pivot(index='modelo',columns='particao',values=['f1_macro','precision','recall']).round(4))

fig, axs = plt.subplots(2,2,figsize=(9,7))
for ax,nome in zip(axs.flat,['aprovacao','despacho','monitor_3','monitor_1']):
    r=finalistas.loc[finalistas.contexto.eq(nome)].iloc[0]
    matriz=np.array([[r.tn,r.fp],[r.fn,r.tp]],dtype=int)
    ax.imshow(matriz,cmap='Blues')
    for (i,j),valor in np.ndenumerate(matriz):
        ax.text(j,i,str(valor),ha='center',va='center',color='white' if valor>matriz.max()/2 else '#202020')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['No prazo','Atrasado'],yticklabels=['No prazo','Atrasado'],
           xlabel='Previsto',ylabel='Observado',title=f"{nome}: {r.modelo}\nF1 macro {r.f1_macro:.3f}")
fig.tight_layout();fig.savefig(FIGURAS/'03_matrizes_confusao.png');plt.show()

### Regiões e distâncias

Cada célula apresenta o F1 macro e o número de pedidos do **mesmo modelo global selecionado**, recortados depois por destino/distância. Esses recortes não treinam outro modelo. Grupos com poucos pedidos/atrasos exigem cautela; melhorar um grupo não demonstra melhoria geral. As linhas de especialistas no CSV permitem comparar a alternativa de treinar modelos separados.

In [ ]:
grupos = resultados.loc[resultados.contexto.eq('aprovacao') & resultados.particao.eq('teste') &
    resultados.selecionado_classificador & resultados.origem_avaliacao.eq('real') &
    resultados.segmento.str.startswith('grupo_regiao_distancia:')].copy()
grupos['grupo']=grupos.segmento.str.split(':',n=1).str[1]
grupos[['regiao','distancia']]=grupos.grupo.str.rsplit('__',n=1,expand=True)
matriz=grupos.pivot(index='regiao',columns='distancia',values='f1_macro').reindex(
    ['SP','Sudeste_sem_SP','Sul','Nordeste','Centro_Oeste','Norte']).reindex(columns=['ate_500','acima_500'])
quant=grupos.pivot(index='regiao',columns='distancia',values='pedidos').reindex_like(matriz)
fig,ax=plt.subplots(figsize=(7,5));img=ax.imshow(matriz,cmap='YlGnBu',vmin=0,vmax=1,aspect='auto')
for i in range(len(matriz)):
    for j in range(2):
        if np.isfinite(matriz.iloc[i,j]):ax.text(j,i,f'{matriz.iloc[i,j]:.3f}\nn={quant.iloc[i,j]:.0f}',ha='center',va='center')
        else:ax.text(j,i,'Sem pedidos',ha='center',va='center',fontsize=9)
ax.set(xticks=[0,1],xticklabels=['Até 500 km','Acima de 500 km'],yticks=range(len(matriz)),yticklabels=matriz.index,
       title='F1 macro no teste real: aprovação, modelo selecionado')
fig.colorbar(img,ax=ax,label='F1 macro');fig.tight_layout();fig.savefig(FIGURAS/'04_regioes_distancias.png');plt.show()
display(grupos[['regiao','distancia','pedidos','pedidos_atrasados','f1_macro','precision','recall']].round(4))

### Sintéticos e mistura: comparação controlada

O gráfico usa somente **teste real comum** e os mesmos 12 atributos. As demais origens de avaliação estão no CSV. O nome `ctgan_real_*` identifica o controle desse experimento: esse candidato foi treinado exclusivamente em dados reais, sem amostras da CTGAN.

In [ ]:
gan_teste=todos.loc[todos.contexto.eq('aprovacao') & todos.particao.eq('teste') & todos.modelo.str.startswith('ctgan_')].copy()
gan_teste['familia']=gan_teste.modelo.str.split('_').str[-1]
fig,ax=plt.subplots(figsize=(8,4))
gan_teste.pivot(index='familia',columns='cenario_treino',values='f1_macro').reindex(columns=['real','sintetico','misto']).plot.bar(
    ax=ax,color=['#32775a','#bd8d41','#537b9b'])
ax.axhline(.5,color='#333333',ls=':');ax.set(ylim=(0,1),xlabel='',ylabel='F1 macro',title='Origem do treino; mesma avaliação real')
ax.tick_params(axis='x',rotation=0);fig.tight_layout();fig.savefig(FIGURAS/'05_ctgan_real_misto.png');plt.show()
display(gan_teste[['modelo','cenario_treino','pedidos','f1_macro','precision','recall','mae_dias','rmse_dias','mae_dias_atrasados']].round(4))

### Sensibilidade ao limiar e incerteza

As curvas de limiar usam somente validação. O intervalo de F1 macro abaixo reamostra semanas do teste, preservando pedidos da mesma semana juntos, com 300 repetições. Ele é descritivo e não corrige a seleção entre muitos candidatos nem garante desempenho futuro; não muda nenhum modelo/limiar.

In [ ]:
meta_por_id={m['id']:m for m in candidatos()}
id_aprovacao=selecao['escolhidos']['aprovacao']['classificador']
meta_aprovacao=meta_por_id[id_aprovacao]
ctx_aprovacao=contexto_avaliacoes(base,'aprovacao')
probs_val=np.load(CACHE/(id_aprovacao+'_scores.npz'))['validacao']
curva=[]
for lim in np.unique(np.r_[np.linspace(0,1,101),meta_aprovacao['limiar']]):
    m=metricas_publicas(ctx_aprovacao['partes']['validacao'].y,probs_val,lim)
    curva.append({k:m[k] for k in ['limiar','f1_macro','precision','recall']})
curva=pd.DataFrame(curva)
fig,ax=plt.subplots(figsize=(8,4));curva.set_index('limiar').plot(ax=ax)
ax.axvline(meta_aprovacao['limiar'],color='black',ls=':',label='Limiar congelado')
ax.set(ylabel='Métrica',title='Aprovação: limiar escolhido somente na validação',ylim=(0,1));ax.legend()
fig.tight_layout();fig.savefig(FIGURAS/'06_limiar_validacao.png');plt.show()

def macro_matriz(y,p):
    y=np.asarray(y,bool);p=np.asarray(p,bool)
    tp=np.sum(y&p);tn=np.sum(~y&~p);fp=np.sum(~y&p);fn=np.sum(y&~p)
    return .5*(2*tp/max(2*tp+fp+fn,1)+2*tn/max(2*tn+fp+fn,1))
intervalos=[];rng=np.random.default_rng(SEED)
for nome in ordem_contextos:
    escolha=selecao['escolhidos'][nome];mid=escolha['classificador']
    ctx=contexto_avaliacoes(base,nome) if nome!='atual_retrospectivo' else construir_contexto(base,nome)
    d=ctx['partes']['teste'];s=np.load(CACHE/(mid+'_teste.npz'))['score'];pred=s>=escolha['limiar'];y=d.y.to_numpy()
    blocos=list(d.groupby(d.momento_predicao.dt.to_period('W')).indices.values())
    valores=[]
    for _ in range(300):
        ix=np.concatenate([blocos[j] for j in rng.integers(0,len(blocos),len(blocos))])
        valores.append(macro_matriz(y[ix],pred[ix]))
    inferior,superior=np.quantile(valores,[.025,.975])
    intervalos.append({'contexto':nome,'f1_macro':macro_matriz(y,pred),'IC95_f1_macro':f'{inferior:.3f} a {superior:.3f}',
                       'semanas':len(blocos),'pedidos':len(d)})
display(pd.DataFrame(intervalos))

### Pacote reutilizável e regressão escolhida

O pacote contém os estimadores completos, atributos e limiar. A função abaixo exemplifica seu consumo com uma tabela já calculada conforme o momento da decisão; ela não adapta o esquema PostgreSQL antigo nem recalcula histórico de eventos. A verificação recarrega o pacote do disco e compara os scores com o experimento.

In [ ]:
def prever_objeto_exportado(obj,cols,d):
    if isinstance(obj,dict):
        if obj['tipo']=='ensemble':
            return sum(w*prever_objeto_exportado(c['modelo'],c['metadados']['atributos'],d)
                for w,c in zip(obj['pesos'],obj['componentes_embutidos']))
        x,_=limpar_x(d,obj['atributos']);p=obj['global'].predict_proba(x)[:,1]
        for grupo,est in obj['especialistas'].items():
            ix=d[obj['grupo']].eq(grupo).to_numpy()
            if ix.any():p[ix]=est.predict_proba(x.loc[ix])[:,1]
        return p
    x,_=limpar_x(d,cols)
    return obj.predict_proba(x)[:,list(obj.classes_).index(1)]

def prever_pacote_exportado(d,pacote):
    with threadpool_limits(limits=THREADS):
        score=prever_objeto_exportado(pacote['classificador'],pacote['atributos_classificador'],d)
        x,_=limpar_x(d,pacote['atributos_regressor']);dias=pacote['regressor'].predict(x)
    return pd.DataFrame({'order_id':d.order_id.to_numpy(),'score_atraso':score,
        'delay_status_previsto':(score>=pacote['limiar']).astype(int),'delay_in_days_previsto':dias,
        'limiar':pacote['limiar']})

pacote=joblib.load(MODELOS_FINAIS/'aprovacao.joblib')
teste_aprovacao=ctx_aprovacao['partes']['teste']
previsoes=prever_pacote_exportado(teste_aprovacao,pacote)
assert np.allclose(previsoes.score_atraso,np.load(CACHE/(id_aprovacao+'_teste.npz'))['score'])
metrica_pacote=metricas_publicas(teste_aprovacao.y,previsoes.score_atraso,pacote['limiar'],
                               teste_aprovacao.delay_in_days,previsoes.delay_in_days_previsto)
display(pd.DataFrame([{'pacote':'aprovacao','pedidos':len(teste_aprovacao),**metrica_pacote}]).round(4))
display(previsoes.head())
fig,ax=plt.subplots(figsize=(6,4.5))
indices=np.random.default_rng(SEED).choice(len(teste_aprovacao),min(5000,len(teste_aprovacao)),replace=False)
ax.scatter(teste_aprovacao.delay_in_days.iloc[indices],previsoes.delay_in_days_previsto.iloc[indices],s=7,alpha=.2,color='#32775a')
minimo=min(ax.get_xlim()[0],ax.get_ylim()[0]);maximo=max(ax.get_xlim()[1],ax.get_ylim()[1]);ax.plot([minimo,maximo],[minimo,maximo],':',color='black')
ax.set(xlabel='Dias observados: entrega − prazo',ylabel='Dias previstos',title='Regressor selecionado: teste real na aprovação')
fig.tight_layout();fig.savefig(FIGURAS/'07_previsao_dias.png');plt.show()

### Mudança temporal e importância dos atributos

A distribuição por mês é apresentada depois do congelamento da seleção. A importância do CatBoost é calculada pelo estimador ajustado no treino; mede uso preditivo dos atributos, não efeito causal. O Olist não fornece versões históricas de todos os campos: assumimos que preço e prazos do registro já eram conhecidos no momento indicado. Essa hipótese precisa ser confirmada na aplicação real.

In [ ]:
operacionais=base.loc[base.operacional].copy()
operacionais['mes']=operacionais.order_approved_at.dt.to_period('M').astype(str)
mensal=operacionais.groupby('mes').agg(pedidos=('y','size'),atrasos=('y','sum'))
mensal['proporcao_atrasos']=mensal.atrasos/mensal.pedidos
fig,ax=plt.subplots(figsize=(10,4));mensal.pedidos.plot.bar(ax=ax,color='#c8d4df',width=.8)
ax2=ax.twinx();ax2.plot(np.arange(len(mensal)),mensal.proporcao_atrasos,color='#a15a38',marker='o',ms=3)
ax.set(ylabel='Pedidos',xlabel='',title='Pedidos e atrasos por mês de aprovação: coorte observada')
ax2.set(ylabel='Proporção de atrasos');fig.tight_layout();fig.savefig(FIGURAS/'08_variacao_temporal.png');plt.show()
display(mensal.tail(10).round(4))
cat=joblib.load(CACHE/'aprovacao__ctgan_real_catboost.joblib')
importancias=pd.Series(cat.get_feature_importance(),index=COLS_GAN).sort_values()
fig,ax=plt.subplots(figsize=(8,4.5));importancias.plot.barh(ax=ax,color='#537b9b')
ax.set(xlabel='Importância calculada pelo CatBoost',title='Aprovação: 12 atributos, treino real')
fig.tight_layout();fig.savefig(FIGURAS/'09_importancia_atributos.png');plt.show()

## 9. Conclusões calculadas a partir dos resultados

As frases seguintes são produzidas pelos dados desta execução. A decisão de usar especialistas, mistura ou mais atributos considera a validação; o teste quantifica como essa decisão se comportou depois.

In [ ]:
r=finalistas.loc[finalistas.contexto.eq('aprovacao')].iloc[0]
b=todos.loc[todos.contexto.eq('aprovacao') & todos.modelo.eq('dummy') & todos.particao.eq('teste')].iloc[0]
meta_atingida=r.f1_macro>=.5
display(Markdown(f"**Meta na aprovação: {'atingida' if meta_atingida else 'não atingida'}.** "
    f"O candidato `{r.modelo}` (treino {r.cenario_treino}) foi escolhido na validação e obteve "
    f"**F1 macro {r.f1_macro:.3f}** no teste real, com {r.pedidos:,} pedidos. "
    f"A referência Dummy teve {b.f1_macro:.3f}. Precision dos alertas: **{r.precision:.1%}**; "
    f"recall dos atrasos: **{r.recall:.1%}**; accuracy/F1 micro: **{r.accuracy:.1%}**. "
    f"Foram {r.tp} atrasos identificados, {r.fn} perdidos e {r.fp} alertas falsos. "
    "A meta macro alcançada não significa que metade dos atrasos tenha sido identificada."))

val=todos.loc[todos.contexto.eq('aprovacao') & todos.particao.eq('validacao')]
global_=val.loc[~val.modelo.str.startswith('especialistas_')].sort_values('f1_macro',ascending=False).iloc[0]
regional=val.loc[val.modelo.str.startswith('especialistas_')].sort_values('f1_macro',ascending=False).iloc[0]
display(Markdown(f"**Separação regional:** melhor especialista na validação: `{regional.modelo}` "
    f"({regional.f1_macro:.3f}); melhor alternativa sem especialistas separados: `{global_.modelo}` "
    f"({global_.f1_macro:.3f}). " + ("Há evidência de ganho de validação com especialistas neste estudo." if regional.f1_macro>global_.f1_macro else
    "Não há ganho de validação que justifique separar o treinamento por região nesta rodada; regiões e distâncias podem continuar como atributos e recortes de avaliação.")))

pares_gan=val.loc[val.modelo.str.startswith('ctgan_')].copy();pares_gan['familia']=pares_gan.modelo.str.split('_').str[-1]
comparacao=pares_gan.pivot(index='familia',columns='cenario_treino',values='f1_macro')
display(comparacao.round(4))
melhor_mistura=(comparacao['misto']-comparacao['real']).max()
display(Markdown("**CTGAN:** "+("a mistura melhorou pelo menos uma família na validação, mas a escolha final continua baseada no teste real reservado." if melhor_mistura>0 else
    "a mistura não superou o controle real em nenhuma das três famílias na validação desta configuração. Não há motivo para acrescentar sintéticos por padrão com estes parâmetros.")+
    " Os resultados não autorizam tratar desempenho em validação/teste sintéticos como prova de melhora na operação real."))

basico=val.loc[val.modelo.eq('catboost_basico')].iloc[0]
compacto=val.loc[val.modelo.eq('ctgan_real_catboost')].iloc[0]
completo=val.loc[val.modelo.eq('catboost')].iloc[0]
display(Markdown(f"**Pipeline e atributos:** no CatBoost, os 10 atributos básicos tiveram F1 macro "
    f"{basico.f1_macro:.3f}; acrescentar os prazos prometido/de despacho resultou em {compacto.f1_macro:.3f}; "
    f"a versão ampla com engenharia e histórico teve {completo.f1_macro:.3f}, todos na mesma validação. "
    "O pipeline garante o mesmo processamento no treino e consumo; o ganho preditivo deve ser atribuído à comparação medida de atributos/modelos, não ao uso da classe Pipeline por si só."))

display(Markdown(f"**Dias de atraso:** o regressor selecionado separadamente teve MAE {metrica_pacote['mae_dias']:.2f} dias, "
    f"RMSE {metrica_pacote['rmse_dias']:.2f}, R² {metrica_pacote['r2_dias']:.3f} e "
    f"MAE de {metrica_pacote['mae_dias_atrasados']:.2f} dias entre os atrasados no teste da aprovação. "
    "Esses erros limitam a utilidade de prever uma data individual. O estudo mede previsão histórica, não atrasos efetivamente evitados por intervenção."))

display(Markdown("**Aplicação ao caso:** usar o modelo escolhido para aprovação como referência inicial e "
    "avaliar despacho/monitoramento como produtos distintos, com suas respectivas coberturas. "
    "Antes de consumo no frontend, é necessário alinhar o contrato de atributos e carregamento do pacote, "
    "manter o histórico disponível no instante da previsão e validar novamente em um período posterior real. "
    "Transportadora precisa ser observada de verdade para medir seu efeito; a coluna aleatória atual serve apenas para demonstração."))

## 10. Integração atual com PostgreSQL, APIs, assistente e frontend

**A nova avaliação e os novos modelos ainda não estão integrados à aplicação.** Esta conclusão vem da inspeção do código, não de uma conexão/teste no PostgreSQL em produção.

| Componente | O que o código usa atualmente | O que falta para o novo fluxo |
|---|---|---|
| `src/postgresql/banco.py` | carga de `Dados/pedidos_ml.csv`, com campos do pipeline anterior | contrato/versionamento para os atributos novos e histórico no instante da decisão |
| `src/postgresql/view_pedidos_inferencia.sql` | view com `valor_itens`, `peso_total_g`, `distancia_media_vendedores_km` etc. | compatibilizar nomes, tipos, semântica e disponibilidade temporal |
| `src/ml/inferencia.py` | `preprocessador_rede.joblib` + `rede_neural.pt`, rede PyTorch anterior | carregar o pacote selecionado, atributos, limiar e regressor compatíveis |
| `src/web/api.py` | `/api/pedidos` e `/api/prioridades` pontuam a validação histórica com essa rede | seleção explícita/versionada do modelo novo e resposta com metadados do modelo |
| Assistente | resumo da fila/scores antigos e contexto dos pedidos | consumir dados/metadados do novo contrato; métricas devem vir do servidor |
| Frontend | consulta os endpoints anteriores | não há tela/endpoint consumindo o novo CSV de desempenho |

A tabela PostgreSQL `avaliacoes` guarda avaliações de clientes; `/api/assistente/avaliacoes` guarda feedback do assistente. Nenhum dos dois representa o CSV de métricas dos modelos. Avaliação offline, previsão de novos pedidos e apresentação no frontend são responsabilidades diferentes.

O ajuste de métricas foi feito em `src/3_avaliar_modelos.py`, usado por `src/2_treinar_modelos.py`; os serviços web e o banco foram preservados. Os CSVs históricos anteriores também foram preservados. Implantação não ocorre automaticamente quando o notebook salva um modelo.

In [ ]:
# Evidência estática e reprodutível: não abre conexão, não inicia API e não envia mensagens.
evidencias=[]
for arquivo,termo in [('src/postgresql/banco.py','pedidos_ml.csv'),
    ('src/postgresql/view_pedidos_inferencia.sql','distancia_media_vendedores_km'),
    ('src/ml/inferencia.py','rede_neural.pt'),('src/ml/inferencia.py','preprocessador_rede.joblib'),
    ('src/web/api.py','return carregar_modelos()'),('src/web/api.py','def contexto_assistente'),
    ('frontend/script.js','/api/prioridades')]:
    texto=(RAIZ/arquivo).read_text(encoding='utf-8')
    ocorrencias=[(i+1,l.strip()) for i,l in enumerate(texto.splitlines()) if termo in l]
    for linha,trecho in ocorrencias:evidencias.append({'arquivo':arquivo,'linha':linha,'trecho':trecho})
display(pd.DataFrame(evidencias))

# Contratos do CSV e preservação dos dados.
salvo=pd.read_csv(CSV_RESULTADOS)
assert set(METRICAS_PUBLICAS).issubset(salvo.columns)
assert not {'f1','roc_auc','average_precision','brier','vies_dias'}.intersection(salvo.columns)
assert not any(c.startswith('fila_') for c in salvo.columns)
assert set(salvo.particao)=={'treino','validacao','teste'}
assert set(salvo.origem_avaliacao)=={'real','sintetico','misto'}
assert all(sha(RAIZ/p)==h for p,h in HASHES.items())
print('Verificado: resultados nas três partições/origens; fontes preservadas; métricas solicitadas; pacote recarregado.')
print('Experimentos de classificação:',len(selecao['candidatos']))
print('Execução:',EXECUCAO)
from importlib.metadata import version
display(pd.DataFrame([{'pacote':p,'versao':version(p)} for p in
    ['numpy','pandas','scikit-learn','catboost','lightgbm','xgboost','imbalanced-learn','ctgan','torch']]))

## 11. Referências consultadas e como foram utilizadas

As referências Olist abaixo são relatos de projetos dos próprios autores, sem equivalência garantida de coorte, alvo, divisão ou métrica. Servem para formular comparações, não para prometer o mesmo resultado.

1. [Jathin — Olist Late Delivery Prediction](https://github.com/jathin2458/olist-late-delivery-prediction): descreve remoção de duração real de transporte/entrega para corrigir vazamento, comparação de classificadores e dificuldade com a classe atrasada. Motivou a auditoria do momento em que cada atributo fica disponível e as referências simples.
2. [Tutusl — Olist Delivery Delay Risk](https://github.com/tutusl/olist-delivery-delay-risk): relata perda de desempenho entre divisão aleatória e avaliação temporal, além de testes em janelas cronológicas. Motivou a comparação temporal. O alvo do projeto usa qualquer entrega posterior ao prazo, enquanto aqui o atraso exige pelo menos um dia; seus números não são diretamente comparáveis.
3. [Ameed2 e equipe — Olist Delivery Delay Prediction](https://github.com/Ameed2/capstone-project3): estuda classificação e previsão de dias; os resultados relatados de regressão não atingem as metas iniciais. Motivou avaliar classificação e duração separadamente. ROC-AUC relatada no projeto não é F1 macro.
4. [Scikit-learn — ajuste do limiar](https://scikit-learn.org/stable/modules/classification_threshold.html): fundamenta separar o score probabilístico da decisão e ajustar o limiar em dados distintos do treino, sem escolher pelo teste.
5. [Scikit-learn — erros comuns e pipelines](https://scikit-learn.org/stable/common_pitfalls.html): orienta ajustar transformações somente no treino e aplicar o mesmo processamento às demais partições.
6. [Imbalanced-learn — vazamento na reamostragem](https://imbalanced-learn.org/stable/common_pitfalls.html): fundamenta manter reamostragem dentro do ajuste do pipeline, com validação/teste reais preservados.

Consulta realizada em 16/11/2026. Nenhum score publicado por terceiros foi tratado como resultado desta execução.